# Persian mT5: summarization + sentiment — Kaggle run

**Before running (right-hand panel → Session options):**
1. **Accelerator:** GPU T4 ×2 or GPU P100 (only one GPU is used).
2. **Internet:** On (needs a phone-verified Kaggle account).

**To run in the background:** click **Save Version → Save & Run All (Commit)**. You can close the browser; results stay in the *Output* tab of the saved version.

**Two runs:**
- Run 1: `RUN_MAIN = True`, `RUN_ABLATION = False` → Stages 1–3, evaluation, figures (about 7–9 h).
- Run 2 (optional): `RUN_MAIN = False`, `RUN_ABLATION = True` → ablation models (about 3–4 h).

In [ ]:
RUN_MAIN = True        # Stage 1 -> 2 -> 3 + evaluation
RUN_ABLATION = False   # sentiment-only and joint-without-curriculum models

## 1. Code and data from GitHub

In [ ]:
import os, glob, zipfile, shutil
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one GPU: effective batch size stays 8, as in the paper
REPO = '/kaggle/working/persian-mt5-summarization-sentiment'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/zph272/persian-mt5-summarization-sentiment.git {REPO}
%cd {REPO}

os.makedirs('data/processed', exist_ok=True)
for zp in glob.glob('*.zip'):
    zipfile.ZipFile(zp).extractall('data/processed')
for f in glob.glob('data/processed/**/*.csv', recursive=True):
    if os.path.dirname(f) != 'data/processed':
        shutil.move(f, 'data/processed/' + os.path.basename(f))
if not os.path.exists('data/processed/joint_clean.csv'):
    shutil.copy('data/processed/joint_train.csv', 'data/processed/joint_clean.csv')
need = ['summ_clean.csv', 'sent_clean.csv', 'joint_train.csv', 'joint_val.csv', 'joint_test.csv']
missing = [f for f in need if not os.path.exists('data/processed/' + f)]
print('MISSING:', missing) if missing else print('All data files are ready')

!pip install -q rouge_score
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)

## 2. Training: Stage 1 → Stage 2 → Stage 3

In [ ]:
if RUN_MAIN:
    !python 4_train_stage1_summ.py

In [ ]:
if RUN_MAIN:
    !python 5_train_stage2_sent.py

In [ ]:
if RUN_MAIN:
    !python 6_train_stage3_joint.py

## 3. Evaluation on the test splits

In [ ]:
if RUN_MAIN:
    !python 7_evaluate.py --baselines
    !python 7_evaluate.py --model summ_final  --tasks summ,loss
    !python 7_evaluate.py --model sent_final  --tasks sent
    !python 7_evaluate.py --model joint_final --tasks summ,sent,loss

## 4. Ablation (Run 2)

In [ ]:
if RUN_ABLATION:
    !python 9_ablation.py

## 5. Figures and results table

In [ ]:
!python 8_plot_results.py
from IPython.display import Image, display, Markdown
if os.path.exists('outputs/results/results_table.md'):
    display(Markdown(open('outputs/results/results_table.md', encoding='utf-8').read()))
for f in sorted(glob.glob('outputs/figures/*.png')):
    print(f); display(Image(f, width=600))

## 6. Results archive
Download `results.zip` from the **Output** tab and send it for the manuscript.

In [ ]:
!zip -qr /kaggle/working/results.zip outputs/results outputs/figures
!ls -lh /kaggle/working/results.zip
!du -sh outputs/models/* 2>/dev/null